# 04 — Lending Club feature pruning

Prepare the **accepted loans** extract for downstream ML by removing identifiers, outcome leakage, and extremely sparse fields. **No model training** in this notebook — only dataset preparation.

**Input:** `datasets/raw/lendingclub/` (auto-resolved CSV)  
**Output:** `datasets/processed/lendingclub_pruned.csv`

## 1. Load dataset (safe path resolution)

In [ ]:
from __future__ import annotations

from pathlib import Path

import pandas as pd


def find_repo_root() -> Path:
    cwd = Path.cwd().resolve()
    for directory in (cwd, *cwd.parents):
        if (directory / "backend").is_dir() and (directory / "ml").is_dir():
            return directory
    return cwd


def resolve_lendingclub_accepted_csv(lc_dir: Path) -> Path:
    """Pick the accepted-loans CSV: prefer canonical name, else first *accepted*.csv."""
    if not lc_dir.is_dir():
        raise FileNotFoundError(
            f"Lending Club directory not found: {lc_dir}. "
            "Create datasets/raw/lendingclub and add the accepted-loans CSV."
        )
    canonical = lc_dir / "accepted_2007_to_2018Q4.csv"
    if canonical.is_file():
        return canonical.resolve()
    accepted = sorted(
        p.resolve() for p in lc_dir.glob("*.csv") if "accepted" in p.name.lower()
    )
    if not accepted:
        listing = sorted(p.name for p in lc_dir.iterdir()) if lc_dir.exists() else []
        raise FileNotFoundError(
            f"No accepted Lending Club CSV in {lc_dir.resolve()}. "
            f"Expected 'accepted_2007_to_2018Q4.csv' or a filename containing 'accepted'. "
            f"Directory contents: {listing or '(empty)'}"
        )
    return accepted[0]


REPO_ROOT = find_repo_root()
LC_DIR = REPO_ROOT / "datasets" / "raw" / "lendingclub"
DATA_PATH = resolve_lendingclub_accepted_csv(LC_DIR)

print(f"Resolved CSV: {DATA_PATH}")

df = pd.read_csv(DATA_PATH, low_memory=False)
print(f"Loaded shape: {df.shape}")

## 2. Basic inspection

In [ ]:
df.shape

In [ ]:
df.info()

In [ ]:
df.head(5)

## 3. Missing value analysis

In [ ]:
missing_count = df.isna().sum()
missing_pct = (missing_count / len(df) * 100).round(4)
missing_table = (
    pd.DataFrame({"column": missing_count.index, "missing_count": missing_count.values, "missing_pct": missing_pct.values})
    .sort_values("missing_count", ascending=False)
    .reset_index(drop=True)
)
missing_table

## 4–6. Prune columns

1. Identifiers / high-cardinality text not useful as features  
2. **Leakage** — post-origination outcomes and servicing fields  
3. **Sparse** — drop columns with **>70%** missing values

Only columns that exist are dropped; all drops are logged.

In [ ]:
IDENTIFIER_COLUMNS = [
    "id",
    "member_id",
    "url",
    "zip_code",
    "title",
    "emp_title",
    "desc",
]

LEAKAGE_COLUMNS = [
    "loan_status",
    "total_pymnt",
    "total_pymnt_inv",
    "total_rec_prncp",
    "total_rec_int",
    "total_rec_late_fee",
    "recoveries",
    "collection_recovery_fee",
    "last_pymnt_d",
    "last_pymnt_amnt",
    "next_pymnt_d",
    "last_credit_pull_d",
    "settlement_status",
    "settlement_date",
    "settlement_amount",
    "settlement_percentage",
    "settlement_term",
]

SPARSE_MISSING_THRESHOLD = 0.70

original_shape = df.shape
original_columns = list(df.columns)
mem_before = int(df.memory_usage(deep=True).sum())

removed: list[str] = []


def drop_if_present(frame: pd.DataFrame, cols: list[str], reason: str) -> pd.DataFrame:
    present = [c for c in cols if c in frame.columns]
    missing = [c for c in cols if c not in frame.columns]
    if missing:
        print(f"[{reason}] not in data (skipped): {missing}")
    if not present:
        print(f"[{reason}] nothing to drop.")
        return frame
    print(f"[{reason}] dropping ({len(present)}): {present}")
    removed.extend(present)
    return frame.drop(columns=present)


df = drop_if_present(df, IDENTIFIER_COLUMNS, "identifiers")
df = drop_if_present(df, LEAKAGE_COLUMNS, "leakage")

missing_rate = df.isna().mean()
sparse_cols = missing_rate[missing_rate > SPARSE_MISSING_THRESHOLD].index.tolist()
print(
    f"[sparse >{SPARSE_MISSING_THRESHOLD:.0%}] columns: {sparse_cols}  |  count: {len(sparse_cols)}"
)
if sparse_cols:
    removed.extend(sparse_cols)
    df = df.drop(columns=sparse_cols)
else:
    print("[sparse] no columns exceeded the missing threshold.")

mem_after = int(df.memory_usage(deep=True).sum())

## 7. Finance / risk column audit (expected to retain)

In [ ]:
USEFUL_FINANCE_COLUMNS = [
    "loan_amnt",
    "term",
    "int_rate",
    "installment",
    "grade",
    "sub_grade",
    "annual_inc",
    "dti",
    "fico_range_low",
    "fico_range_high",
    "delinq_2yrs",
    "inq_last_6mths",
    "open_acc",
    "pub_rec",
    "revol_bal",
    "revol_util",
    "home_ownership",
    "verification_status",
    "purpose",
    "application_type",
    "emp_length",
]

audit = pd.DataFrame(
    {
        "column": USEFUL_FINANCE_COLUMNS,
        "present_after_prune": [c in df.columns for c in USEFUL_FINANCE_COLUMNS],
    }
)
print("Present (True) vs missing from pruned frame (False):")
audit

## 8. Final summary

In [ ]:
def _fmt_mem(n: int) -> str:
    return f"{n / (1024 ** 2):.2f} MiB ({n:,} bytes)"


final_columns = list(df.columns)
retained_set = set(final_columns)
original_set = set(original_columns)
removed_set = original_set - retained_set

print("=== Pruning summary ===")
print(f"Original shape: {original_shape}")
print(f"Final shape:    {df.shape}")
print(f"Columns removed ({len(removed_set)}): {sorted(removed_set)}")
print(f"Columns retained ({len(retained_set)}): {sorted(retained_set)}")
print(f"Memory (deep estimate) before: {_fmt_mem(mem_before)}")
print(f"Memory (deep estimate) after:  {_fmt_mem(mem_after)}")

## 9. Save cleaned interim dataset

In [ ]:
OUTPUT_PATH = REPO_ROOT / "datasets" / "processed" / "lendingclub_pruned.csv"
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)

df.to_csv(OUTPUT_PATH, index=False)
print(f"Wrote: {OUTPUT_PATH.resolve()}  |  shape: {df.shape}")